# Wind Forecast Evaluation: AIFS vs IFS
## Analysis of Onshore, Coastal, and Offshore Locations around Ireland

**Updated for ecmwf_forecasts data structure**

This notebook implements the 8-step evaluation plan:
1. Select locations (onshore, coastal, offshore)
2. Extract forecasts from downloaded data
3. Ensure consistent alignment (time, height, units)
4. Compute 10m wind speeds from components
5. Define extreme wind events
6. Evaluate deterministic forecast performance
7. Analyze AIFS ensemble behavior
8. Compare performance across environments

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import our utilities
from wind_utils import DataLoader, WindMetrics, ForecastVerification
from config_updated import (
    LOCATIONS, MODEL_CONFIGS, LEAD_TIMES, INIT_TIMES,
    get_forecast_files, list_available_dates, list_available_times,
    EXTREME_THRESHOLDS, MODEL_COLORS, ENVIRONMENT_COLORS
)

# Check for cfgrib
try:
    import cfgrib
    print("✓ cfgrib available")
except ImportError:
    print("Installing cfgrib...")
    !pip install cfgrib --break-system-packages
    import cfgrib

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully!")

## Step 0: Check Available Data

First, let's see what forecast data we have downloaded.

In [ ]:
# Check available dates
available_dates = list_available_dates()

print("="*60)
print("AVAILABLE FORECAST DATA")
print("="*60)
print(f"\nDate offsets found: {available_dates}")

if not available_dates:
    print("\n⚠️  No forecast data found!")
    print("   Run the download script first.")
else:
    # Check each model
    for model_key, config in MODEL_CONFIGS.items():
        print(f"\n{config['display_name']}:")
        for date_offset_str in available_dates:
            date_offset = int(date_offset_str.replace('date', ''))
            times = list_available_times(model_key, date_offset)
            if times:
                print(f"  {date_offset_str}: {', '.join(times)}")
    
    print(f"\n✓ Ready to proceed with analysis!")

## Step 1: Select Locations

We'll analyze locations across different environments.

In [ ]:
# Display locations
locations_df = pd.DataFrame(LOCATIONS).T

print("="*60)
print("ANALYSIS LOCATIONS")
print("="*60)
print(f"\nTotal locations: {len(LOCATIONS)}")
print(f"  Onshore:  {sum(locations_df['type'] == 'onshore')}")
print(f"  Coastal:  {sum(locations_df['type'] == 'coastal')}")
print(f"  Offshore: {sum(locations_df['type'] == 'offshore')}")

print("\nLocation Details:")
print(locations_df[['name', 'lat', 'lon', 'type']])

## Step 2-3: Load and Align Forecast Data

Load forecasts from the ecmwf_forecasts directory.

In [ ]:
# Select one location for detailed analysis
example_location = 'Malin_Head'
lat = LOCATIONS[example_location]['lat']
lon = LOCATIONS[example_location]['lon']

print(f"Analyzing: {LOCATIONS[example_location]['name']}")
print(f"Coordinates: {lat:.4f}°N, {lon:.4f}°E")
print(f"Type: {LOCATIONS[example_location]['type']}")

# Choose which date/time to analyze
if available_dates:
    date_offset = int(available_dates[0].replace('date', ''))  # Most recent
    init_time = '00z'  # Can change to 06z, 12z, 18z
    
    print(f"\nAnalyzing: {available_dates[0]}, {init_time} initialization")
else:
    print("\n⚠️  No data available. Please run download script.")

In [ ]:
# Load forecasts for all lead times
loader = DataLoader()
forecasts = {}

for model_key in ['ifs', 'aifs_single']:
    model_name = MODEL_CONFIGS[model_key]['display_name']
    forecasts[model_name] = {}
    
    print(f"\nLoading {model_name}:")
    
    for lead_time in LEAD_TIMES:
        try:
            data = loader.load_forecast_from_new_structure(
                model_key=model_key,
                date_offset=date_offset,
                init_time=init_time,
                lead_time=lead_time,
                lat=lat,
                lon=lon
            )
            
            if data and 'wind_speed' in data:
                forecasts[model_name][lead_time] = data
                print(f"  F{lead_time:03d}h: {data['wind_speed']:.2f} m/s ✓")
            else:
                print(f"  F{lead_time:03d}h: Not available ✗")
                
        except Exception as e:
            print(f"  F{lead_time:03d}h: Error - {e}")

print(f"\n✓ Loaded forecasts for {len(forecasts)} models")

## Step 4: Compute Wind Speeds

Wind speeds are already computed in the loading step.
Let's visualize them.

In [ ]:
# Create wind speed comparison plot
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, model_data in forecasts.items():
    if model_data:
        lead_times = sorted(model_data.keys())
        wind_speeds = [model_data[lt]['wind_speed'] for lt in lead_times]
        
        ax.plot(lead_times, wind_speeds, 'o-', 
               label=model_name, linewidth=2, markersize=8)

ax.set_xlabel('Forecast Lead Time (hours)', fontsize=12)
ax.set_ylabel('Wind Speed (m/s)', fontsize=12)
ax.set_title(f'Wind Speed Forecasts: {LOCATIONS[example_location]["name"]}',
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(f'./wind_speed_forecast_{example_location}.png', 
           dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved")

## Step 5: Define Extreme Wind Events

Classify winds by Beaufort scale and define extremes.

In [ ]:
wind_metrics = WindMetrics()

# Beaufort scale thresholds
beaufort_thresholds = {
    'Moderate breeze': 5.5,
    'Fresh breeze': 8.0,
    'Strong breeze': 10.8,
    'Near gale': 13.9,
    'Gale': 17.2,
    'Strong gale': 20.8,
    'Storm': 24.5
}

print("="*60)
print("WIND CLASSIFICATION")
print("="*60)

for model_name, model_data in forecasts.items():
    if model_data:
        print(f"\n{model_name}:")
        for lead_time in sorted(model_data.keys()):
            ws = model_data[lead_time]['wind_speed']
            
            # Classify
            category = 'Calm'
            for name, threshold in beaufort_thresholds.items():
                if ws >= threshold:
                    category = name
            
            print(f"  F{lead_time:03d}h: {ws:5.2f} m/s ({category})")

# Check for extreme events
gale_threshold = EXTREME_THRESHOLDS['gale']
print(f"\n{'='*60}")
print(f"Gale force threshold: {gale_threshold} m/s")
print(f"{'='*60}")

## Step 6: Compare Models

Direct comparison of AIFS vs IFS at each lead time.

In [ ]:
# Create comparison table
comparison_data = []

for lead_time in LEAD_TIMES:
    row = {'Lead Time (h)': lead_time}
    
    for model_name in forecasts.keys():
        if lead_time in forecasts[model_name]:
            ws = forecasts[model_name][lead_time]['wind_speed']
            row[model_name] = f"{ws:.2f}"
        else:
            row[model_name] = 'N/A'
    
    # Calculate difference
    if 'IFS' in forecasts and 'AIFS Single' in forecasts:
        if lead_time in forecasts['IFS'] and lead_time in forecasts['AIFS Single']:
            ifs_ws = forecasts['IFS'][lead_time]['wind_speed']
            aifs_ws = forecasts['AIFS Single'][lead_time]['wind_speed']
            diff = aifs_ws - ifs_ws
            row['Difference'] = f"{diff:+.2f}"
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("="*60)
print(f"MODEL COMPARISON: {LOCATIONS[example_location]['name']}")
print("="*60)
print(comparison_df.to_string(index=False))
print("\nUnits: m/s")
print("Difference = AIFS - IFS")

## Step 7: Analyze Multiple Locations

Loop through all locations and collect statistics.

In [ ]:
# Analyze all locations
all_results = {}

print("Analyzing all locations...\n")

for loc_name, loc_info in LOCATIONS.items():
    print(f"Processing: {loc_name}...", end=" ")
    
    loc_results = {
        'location': loc_name,
        'type': loc_info['type'],
        'lat': loc_info['lat'],
        'lon': loc_info['lon']
    }
    
    # Load forecasts for this location
    for model_key in ['ifs', 'aifs_single']:
        model_name = MODEL_CONFIGS[model_key]['display_name']
        
        for lead_time in LEAD_TIMES:
            try:
                data = loader.load_forecast_from_new_structure(
                    model_key=model_key,
                    date_offset=date_offset,
                    init_time=init_time,
                    lead_time=lead_time,
                    lat=loc_info['lat'],
                    lon=loc_info['lon']
                )
                
                if data and 'wind_speed' in data:
                    key = f"{model_name}_F{lead_time:03d}h"
                    loc_results[key] = data['wind_speed']
                    
            except Exception as e:
                pass
    
    all_results[loc_name] = loc_results
    print("✓")

# Convert to DataFrame
results_df = pd.DataFrame(all_results).T

print(f"\n✓ Analyzed {len(all_results)} locations")

In [ ]:
# Display results by environment type
print("="*80)
print("RESULTS BY ENVIRONMENT TYPE")
print("="*80)

for env_type in ['onshore', 'coastal', 'offshore']:
    print(f"\n{env_type.upper()}:")
    env_results = results_df[results_df['type'] == env_type]
    
    if len(env_results) > 0:
        # Get wind speed columns
        ws_cols = [col for col in results_df.columns if 'F0' in str(col)]
        
        print(env_results[['location'] + ws_cols].to_string())
    else:
        print("  No data available")

## Step 8: Environmental Comparison

Compare forecast performance across environments.

In [ ]:
# Calculate statistics by environment
env_stats = []

for env_type in ['onshore', 'coastal', 'offshore']:
    env_data = results_df[results_df['type'] == env_type]
    
    if len(env_data) == 0:
        continue
    
    # Calculate mean wind speed for each model/lead time
    for model_name in ['IFS', 'AIFS Single']:
        for lead_time in LEAD_TIMES:
            col = f"{model_name}_F{lead_time:03d}h"
            if col in env_data.columns:
                values = pd.to_numeric(env_data[col], errors='coerce')
                if not values.isna().all():
                    env_stats.append({
                        'Environment': env_type.capitalize(),
                        'Model': model_name,
                        'Lead Time': f"{lead_time}h",
                        'Mean Wind Speed': values.mean(),
                        'Std Dev': values.std(),
                        'N Locations': values.notna().sum()
                    })

env_stats_df = pd.DataFrame(env_stats)

print("="*80)
print("ENVIRONMENTAL COMPARISON")
print("="*80)
print(env_stats_df.to_string(index=False))
print("\nUnits: m/s")

In [ ]:
# Create environmental comparison plot
if len(env_stats_df) > 0:
    fig, axes = plt.subplots(1, len(LEAD_TIMES), figsize=(15, 5))
    
    if len(LEAD_TIMES) == 1:
        axes = [axes]
    
    for idx, lead_time in enumerate(LEAD_TIMES):
        ax = axes[idx]
        
        # Filter data for this lead time
        lt_data = env_stats_df[env_stats_df['Lead Time'] == f"{lead_time}h"]
        
        if len(lt_data) > 0:
            # Pivot for plotting
            pivot = lt_data.pivot(index='Environment', 
                                 columns='Model', 
                                 values='Mean Wind Speed')
            
            pivot.plot(kind='bar', ax=ax, width=0.8)
            ax.set_title(f'Lead Time: {lead_time}h', fontsize=12, fontweight='bold')
            ax.set_xlabel('Environment', fontsize=11)
            ax.set_ylabel('Mean Wind Speed (m/s)', fontsize=11)
            ax.legend(title='Model')
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.savefig('./environmental_comparison.png', 
               dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Environmental comparison plot saved")
else:
    print("\n⚠️  Not enough data for environmental comparison plot")

## Summary and Conclusions

In [ ]:
print("="*80)
print("ANALYSIS SUMMARY")
print("="*80)

print(f"\nData analyzed:")
print(f"  Date: {available_dates[0]}")
print(f"  Initialization: {init_time}")
print(f"  Lead times: {LEAD_TIMES} hours")

print(f"\nLocations analyzed: {len(all_results)}")
print(f"  Onshore:  {sum(results_df['type'] == 'onshore')}")
print(f"  Coastal:  {sum(results_df['type'] == 'coastal')}")
print(f"  Offshore: {sum(results_df['type'] == 'offshore')}")

print(f"\nModels compared:")
for model in forecasts.keys():
    print(f"  • {model}")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("\n1. Download more historical data for statistical analysis")
print("2. Add observational data (ERA5, station obs) for verification")
print("3. Calculate verification metrics (RMSE, bias, correlation)")
print("4. Analyze extreme wind events")
print("5. Compare ensemble spread (when AIFS ensemble data available)")

print("\n✓ Analysis complete!")